In [1]:
import os
import seaborn as sns
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [22]:
## FUNCTIONS

## GET DATA from parent intake interview

def get_data(preprocess=False, target_col='DX_01'):

    import pandas as pd
    from hbn.features import build_features
    from hbn.data import make_dataset

    # get features
    features = build_features.get_features(
                assessment='Child Measures',
                domains=['Questionnaire Measures of Substance Use & Addiction'],
                measures=['Internet Addiction Test'],
                min_num_participants=2000,
                incl_data_type=None
                );

    if preprocess:
        # preprocess features 
        clf_info={"numeric": [[
                        "sklearn.impute",
                        "SimpleImputer",
                        {"strategy": "mean"}]
                ]
                }
        features = build_features.preprocess(
                                            dataframe=features,
                                            clf_info=clf_info,
                                            cols_to_ignore=['Identifiers']
                                            )

    # get targets
    targets = build_features.get_targets(target_info = {
                                        "assessment": "Clinical Measures",
                                        "domain": None,
                                        "measure": "Clinical Diagnosis Demographics",
                                        "target_column": target_col,
                                        "transform": None,
                                        "outname":target_col
                                        })
    # get participant ids
    participants = make_dataset.get_participants(
                                split='all', 
                                disorders=['ADHD-Combined Type', 
                                            'ADHD-Inattentive Type', 
                                            'ADHD-Hyperactive_Impulsive_Type', 
                                            'No_Diagnosis_Given']
                                )

    features_target = features.merge(
                                targets, on='Identifiers').merge(
                                participants, on='Identifiers')
    # get feature names
    feature_names = [col for col in features_target.columns if target_col not in col]
    
    df_concat = pd.concat([features_target[[target_col]], features_target[feature_names]], axis=1)

    return df_concat

In [23]:
from hbn.data import make_dataset

# get summary of clinical diagnosis + other demographics
dx = make_dataset.make_summary(save=False)
dx = make_dataset._add_race_ethnicity(dataframe=dx)

# filter for adhd
adhd_only = ['ADHD-Combined Type', 'ADHD-Hyperactive/Impulsive Type', 'ADHD-Inattentive Type', 'No Diagnosis Given']
dx = dx[dx['DX_01'].isin(adhd_only)]

# get data from intake interview and merge with clinical summary
df = get_data()
df = df.merge(dx[['Sex', 'Age_bracket', 'PreInt_Demos_Fam,Child_Race_cat','Identifiers']], on='Identifiers')


reading /Users/maedbhking/Documents/healthy_brain_network/data/raw/phenotype/Child_Measures/Questionnaire_Measures_of_Substance_Use_&_Addiction/Internet_Addiction_Test.csv into dataframe
reading /Users/maedbhking/Documents/healthy_brain_network/data/raw/phenotype/Clinical_Measures/Clinical_Diagnosis_Demographics.csv into dataframe


In [24]:
df

,DX_01,Identifiers,"IAT,Administration","IAT,Data_entry","IAT,Days_Baseline","IAT,EID","IAT,IAT_01","IAT,IAT_02","IAT,IAT_03","IAT,IAT_04",...,"WIAT,WIAT_Spell_Raw","WIAT,WIAT_Spell_Stnd","WIAT,WIAT_Valid","WIAT,WIAT_Word_P","WIAT,WIAT_Word_Raw","WIAT,WIAT_Word_Stnd","WIAT,Year",Sex,Age_bracket,"PreInt_Demos_Fam,Child_Race_cat"
0,ADHD-Combined Type,NDARBM433VER,All,Complete,0.0,NDARBM433VER,5.0,0.0,0.0,5.0,...,33.0,117.0,1.0,82.0,54.0,114.0,2018.0,male,under10,White/Caucasian
1,No Diagnosis Given,NDARCD401HGZ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,54.0,115.0,NaN,90.0,74.0,119.0,2016.0,male,over10,Unknown
2,ADHD-Inattentive Type,NDARCM811CV1,All,Complete,114.0,NDARCM811CV1,5.0,4.0,1.0,4.0,...,54.0,115.0,1.0,86.0,73.0,116.0,2018.0,male,over10,Other race
3,ADHD-Inattentive Type,NDARCP292KPA,All,Complete,0.0,NDARCP292KPA,2.0,3.0,0.0,2.0,...,59.0,120.0,1.0,81.0,72.0,113.0,2018.0,male,over10,Unknown
4,ADHD-Combined Type,NDARDX770PJK,All,Complete,0.0,NDARDX770PJK,4.0,2.0,1.0,1.0,...,54.0,110.0,NaN,77.0,71.0,111.0,2016.0,female,over10,White/Caucasian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1862,No Diagnosis Given,NDARYR771VED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.0,108.0,1.0,NaN,NaN,NaN,2016.0,female,under10,White/Caucasian
1863,ADHD-Inattentive Type,NDARYV654UGB,All,Complete,56.0,NDARYV654UGB,0.0,0.0,0.0,0.0,...,11.0,90.0,1.0,63.0,27.0,105.0,2020.0,male,under10,Black/African American
1864,ADHD-Hyperactive/Impulsive Type,NDARYZ433UL5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,28.0,150.0,1.0,99.0,50.0,148.0,2020.0,male,under10,White/Caucasian
1865,ADHD-Inattentive Type,NDARZL162HZH,All,Complete,91.0,NDARZL162HZH,0.0,0.0,0.0,1.0,...,14.0,75.0,1.0,7.0,26.0,78.0,2020.0,male,over10,White/Caucasian


In [28]:
colors = ['Sex']

variable = 'IAT,IAT_08'

for color in colors:
    tmp = df.groupby(['DX_01', color]).agg({variable: 'median',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp[variable] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y=variable, color=color,orientation='v', barmode="group")
    fig.show()

In [29]:
tmp

,DX_01,Sex,"IAT,IAT_08",Identifiers,percent
0,ADHD-Combined Type,female,0.0,172,0.0
1,ADHD-Combined Type,male,0.0,581,0.0
2,ADHD-Hyperactive/Impulsive Type,female,0.0,29,0.0
3,ADHD-Hyperactive/Impulsive Type,male,0.0,77,0.0
4,ADHD-Inattentive Type,female,0.0,218,0.0
5,ADHD-Inattentive Type,male,0.0,462,0.0
6,No Diagnosis Given,female,0.0,158,0.0
7,No Diagnosis Given,male,0.0,170,0.0
